# WDMamba Phase 1: baseline reproduction and profiling

This notebook is self-contained for Google Colab. It clones WDMamba, creates an isolated Python environment, downloads the RESIDE-6K test split and the published checkpoint, then measures PSNR/SSIM, latency/FPS, VRAM, FLOPs, and the native DWT/LFRN/IWT/DEN stages.

Run the cells in order. Start with `LIMIT = 16` as a smoke test; after it succeeds, set `LIMIT = 0` and rerun the benchmark cell for the complete test split. The last cell creates a ZIP containing every required Phase 1 report.

## 1. Clone repository và cấu hình

Notebook có thể được mở trực tiếp từ GitHub hoặc tải lên Colab. Nếu repo đã tồn tại,
cell này tái sử dụng checkout; không reset hay ghi đè thay đổi trong repo.


In [ ]:
from pathlib import Path
import json
import platform
import subprocess
import sys

REPO_URL = "https://github.com/Darkin72/WDMamba.git"
REPO_ROOT = Path("/content/WDMamba_phase1")
WORK_ROOT = Path("/content/wdmamba_phase1")
OUTPUT_ROOT = WORK_ROOT / "results"
ENV_ROOT = Path("/content/wdmamba_phase1_python")
PYTHON = ENV_ROOT / "bin/python"

def run_command(args, cwd=None):
    args = [str(arg) for arg in args]
    print("$", " ".join(args), flush=True)
    subprocess.run(args, cwd=cwd, check=True)

if platform.system() != "Linux" or not Path("/content").is_dir():
    raise RuntimeError("This notebook is intended for Google Colab/Linux.")
if not REPO_ROOT.exists():
    run_command(["git", "clone", "--depth", "1", REPO_URL, REPO_ROOT])
if not (REPO_ROOT / "basicsr/archs/wavemamba_arch.py").is_file():
    raise FileNotFoundError("The checkout does not contain the WDMamba architecture.")
run_command(["git", "log", "-1", "--oneline"], cwd=REPO_ROOT)

BATCH_SIZE = 1
MAX_SIDE = 0
LIMIT = 16
WARMUP = 10
TIMING_REPEATS = 30
NUM_WORKERS = 2
PROFILE_SIZE = 256
SEED = 666
CONFIG = {
    "repo_root": str(REPO_ROOT), "work_root": str(WORK_ROOT),
    "output_root": str(OUTPUT_ROOT), "batch_size": BATCH_SIZE,
    "max_side": MAX_SIDE, "limit": LIMIT, "warmup": WARMUP,
    "timing_repeats": TIMING_REPEATS, "num_workers": NUM_WORKERS,
    "profile_size": PROFILE_SIZE, "seed": SEED,
    "paper_psnr": 32.15, "paper_ssim": 0.9723,
    "psnr_tolerance": 0.50, "ssim_tolerance": 0.005,
    "den_share_threshold": 0.50,
}
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = WORK_ROOT / "config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
print(CONFIG)

## 2. Isolated CUDA/Mamba environment

The environment uses PyTorch 2.8.0 with CUDA 12.8 and builds `causal-conv1d`/`mamba-ssm` from source. This avoids relying on an older CUDA wheel when running on Blackwell.

In [ ]:
run_command(["nvidia-smi"])
run_command([sys.executable, "-m", "pip", "install", "-q", "uv"])
run_command([sys.executable, "-m", "uv", "python", "install", "3.11"])
if not PYTHON.is_file():
    run_command([sys.executable, "-m", "uv", "venv", "--python", "3.11", ENV_ROOT])

TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu128"
run_command([sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON,
             "--index-url", TORCH_INDEX_URL, "torch==2.8.0", "torchvision==0.23.0"])
constraints = ENV_ROOT / "constraints.txt"
constraints.write_text("torch==2.8.0\ntorchvision==0.23.0\nnumpy<2\nsetuptools<81\n", encoding="utf-8")
packages = ["numpy<2", "opencv-python-headless<4.12", "scipy", "scikit-image", "pandas",
            "einops", "timm==1.0.15", "pytorch-wavelets", "PyWavelets", "setuptools<81",
            "transformers", "gdown", "kagglehub", "thop", "tqdm", "pyiqa", "pyyaml"]
run_command([sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON,
             "--constraint", constraints, *packages])
run_command([sys.executable, "-m", "uv", "pip", "install", "--python", PYTHON,
             "--constraint", constraints, "--no-build-isolation", "causal-conv1d", "mamba-ssm"])

def run_python(label, source):
    script = WORK_ROOT / f"{label}.py"
    script.write_text(source.strip() + "\n", encoding="utf-8")
    run_command([PYTHON, "-u", script, CONFIG_PATH], cwd=REPO_ROOT)

run_python("check_blackwell_environment", r"""
import json, sys
from pathlib import Path
import torch
cfg = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
assert torch.cuda.is_available(), "CUDA GPU is required."
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability())
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
with torch.inference_mode():
    u = torch.zeros(1, 4, 16, device="cuda")
    delta = torch.ones_like(u)
    A = -torch.ones(4, device="cuda")
    B = torch.ones(1, 1, 16, device="cuda")
    C = torch.ones(1, 1, 16, device="cuda")
    y = selective_scan_fn(u, delta, A, B, C, delta_softplus=True)
    assert torch.isfinite(y).all()
print("Selective scan smoke test: PASS")
""")

## 3. RESIDE-6K test split

Set `LOCAL_DATA_ROOT` only when the dataset is already available in Drive. Otherwise KaggleHub downloads `kmljts/reside-6k`; Kaggle credentials may be required by Colab.

In [ ]:
LOCAL_DATA_ROOT = ""
KAGGLE_DATASET = "kmljts/reside-6k"
CONFIG["local_data_root"] = LOCAL_DATA_ROOT
CONFIG["kaggle_dataset"] = KAGGLE_DATASET
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
run_python("prepare_reside6k", r"""
import json, sys
from pathlib import Path
import kagglehub
cfg = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
downloaded = Path(cfg["local_data_root"]) if cfg.get("local_data_root") else Path(kagglehub.dataset_download(cfg["kaggle_dataset"]))

def find_pair_root(base):
    candidates = [base, *[p for p in base.rglob("*") if p.is_dir()]]
    candidates.sort(key=lambda p: (0 if "test" in {part.lower() for part in p.parts} else 1, len(p.parts)))
    for candidate in candidates:
        dirs = {p.name.lower(): p for p in candidate.iterdir() if p.is_dir()}
        gt = next((dirs[k] for k in ("clear", "gt", "clean", "groundtruth") if k in dirs), None)
        lq = next((dirs[k] for k in ("hazy", "haze", "input", "inputs") if k in dirs), None)
        if gt is not None and lq is not None:
            parts = {part.lower() for part in candidate.parts}
            if "test" not in parts and any(token in parts for token in ("train", "training", "val", "validation")):
                continue
            return candidate, gt, lq
    raise FileNotFoundError(f"Could not find the RESIDE-6K test pair below {base}")

base, gt, lq = find_pair_root(downloaded)
manifest = {"dataset_root": str(downloaded.resolve()), "datasets": {"RESIDE-6K-test": str(base.resolve())},
            "clear_dir": str(gt.resolve()), "hazy_dir": str(lq.resolve())}
path = Path(cfg["work_root"]) / "inputs.json"
path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
""")

## 4. Published WDMamba checkpoint

The default Google Drive archive is the checkpoint used for the RESIDE-6K result. Set `LOCAL_ZIP` when the archive has already been downloaded.

In [ ]:
LOCAL_ZIP = ""
CONFIG["local_zip"] = LOCAL_ZIP
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
run_python("download_weights", r"""
import json, sys, shutil, zipfile
from pathlib import Path
import gdown
cfg = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
root = Path(cfg["work_root"]) / "checkpoints"
root.mkdir(parents=True, exist_ok=True)
target = root / "WDMamba_ckpts.zip"
if not target.is_file() or not zipfile.is_zipfile(target):
    partial = target.with_suffix(".zip.partial")
    if cfg.get("local_zip"):
        shutil.copy2(cfg["local_zip"], partial)
    else:
        result = gdown.download(id="1Vm-1mtMSc9YIUICq4nWKu-4AHCtYnPpF", output=str(partial), quiet=False, use_cookies=False, resume=True)
        if result is None:
            raise RuntimeError("Checkpoint download failed; set LOCAL_ZIP to a local archive.")
    with zipfile.ZipFile(partial) as archive:
        if archive.testzip() is not None:
            raise RuntimeError("Checkpoint archive is corrupt.")
    partial.replace(target)
extract_root = root / "unzipped"
matches = list(extract_root.rglob("reside6k_32.15.pth")) if extract_root.exists() else []
if not matches:
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(target) as archive:
        for member in archive.infolist():
            resolved = (extract_root / member.filename).resolve()
            if not resolved.is_relative_to(extract_root.resolve()):
                raise ValueError("Checkpoint archive contains an unsafe path.")
        archive.extractall(extract_root)
    matches = list(extract_root.rglob("reside6k_32.15.pth"))
if len(matches) != 1:
    raise FileNotFoundError(f"Expected one reside6k_32.15.pth, found: {matches}")
manifest_path = Path(cfg["work_root"]) / "inputs.json"
data = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
data["checkpoint"] = str(matches[0].resolve())
manifest_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
print("Checkpoint:", matches[0])
""")

## 5. Benchmark runner

The runner verifies that the staged DWT/LFRN/IWT/DEN path matches the original WDMamba forward pass, then records quality, timing, memory, and FLOPs. Measurements are saved after each batch.

In [ ]:
import ast
import textwrap
RUNNER_SOURCE = textwrap.dedent('\nimport ast\nimport importlib.machinery\nimport importlib.util\nimport json\nimport math\nimport re\nimport subprocess\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom skimage.metrics import peak_signal_noise_ratio, structural_similarity\nfrom thop import profile\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\nfrom tqdm import tqdm\n\nCFG = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))\nINPUTS = json.loads((Path(CFG["work_root"]) / "inputs.json").read_text(encoding="utf-8"))\nROOT = Path(CFG["repo_root"])\nOUTPUT = Path(CFG["output_root"])\nassert CFG["batch_size"] >= 1\nassert CFG["warmup"] >= 1 and CFG["timing_repeats"] >= 1\nassert torch.cuda.is_available(), "Cần GPU CUDA."\ntorch.backends.cudnn.benchmark = True\ntorch.backends.cuda.matmul.allow_tf32 = False\ntorch.backends.cudnn.allow_tf32 = False\ntorch.manual_seed(666)\ncv2.setNumThreads(0)\n\ndef import_architecture():\n    # Tránh __init__ tự động import toàn bộ training stack.\n    for name, directory in (\n        ("basicsr", ROOT / "basicsr"),\n        ("basicsr.utils", ROOT / "basicsr/utils"),\n        ("basicsr.archs", ROOT / "basicsr/archs"),\n    ):\n        if name in sys.modules:\n            raise RuntimeError(f"Package đã được import từ trước: {name}")\n        spec = importlib.machinery.ModuleSpec(name, loader=None, is_package=True)\n        spec.submodule_search_locations = [str(directory)]\n        sys.modules[name] = importlib.util.module_from_spec(spec)\n    from basicsr.archs.wavemamba_arch import WaveMamba, SS2D\n    from basicsr.archs.detail_enhance_net import DENet, fft_bench\n    from basicsr.archs.wavelet import DWT, IWT\n    from pytorch_wavelets import DWTForward\n    return WaveMamba, SS2D, DENet, fft_bench, DWT, IWT, DWTForward\n\nWaveMamba, SS2D, DENet, FFTBench, DWT, IWT, DWTForward = import_architecture()\n\ndef load_model():\n    state = torch.load(INPUTS["checkpoint"], map_location="cpu", weights_only=False)\n    if isinstance(state, dict):\n        for key in ("params", "state_dict", "model"):\n            if key in state:\n                state = state[key]\n                break\n    state = {key.removeprefix("module."): value for key, value in state.items()}\n    groups = {i: set() for i in (1, 2, 3)}\n    for key in state:\n        match = re.fullmatch(r"restoration_network\\.DE\\.g([123])\\.gp\\.(\\d+)\\.conv1\\.weight", key)\n        if match:\n            groups[int(match[1])].add(int(match[2]))\n    counts = {len(v) for v in groups.values()}\n    if len(counts) != 1 or next(iter(counts)) not in (4, 6):\n        raise ValueError(f"Không nhận diện được số block DEN: {groups}")\n    blocks = next(iter(counts))\n    assert all(v == set(range(blocks)) for v in groups.values())\n    model = WaveMamba(in_chn=3, wf=16, n_l_blocks=[1, 2, 2, 4], ffn_scale=2.0)\n    if blocks == 4:\n        model.restoration_network.DE = DENet(3, 4)\n    model.load_state_dict(state, strict=True)\n    return model.cuda().eval(), blocks\n\nEXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}\ndef files(root):\n    return sorted(p for p in root.iterdir() if p.is_file() and p.suffix.lower() in EXT)\n\ndef collect_pairs(root):\n    clear = {}\n    for path in files(root / "clear"):\n        if path.stem in clear:\n            raise ValueError(f"Trùng tên ảnh sạch: {path.stem}")\n        clear[path.stem] = path\n    pairs = []\n    for hazy in files(root / "hazy"):\n        gt = clear.get(hazy.stem, clear.get(hazy.stem.split("_")[0]))\n        if gt is None:\n            raise FileNotFoundError(f"Không có ảnh sạch tương ứng với {hazy}")\n        pairs.append((hazy.stem, hazy, gt))\n    if CFG["limit"] > 0:\n        pairs = pairs[:CFG["limit"]]\n    if not pairs:\n        raise ValueError(f"Không có cặp ảnh dưới {root}")\n    return pairs\n\nclass PairedDataset(Dataset):\n    def __init__(self, pairs):\n        self.pairs = pairs\n    def __len__(self):\n        return len(self.pairs)\n    def __getitem__(self, index):\n        name, hazy_path, gt_path = self.pairs[index]\n        hazy, gt = [cv2.imread(str(p), cv2.IMREAD_COLOR) for p in (hazy_path, gt_path)]\n        if hazy is None or gt is None:\n            raise RuntimeError(f"Không đọc được cặp ảnh {name}")\n        hazy, gt = [cv2.cvtColor(img, cv2.COLOR_BGR2RGB) for img in (hazy, gt)]\n        if hazy.shape[:2] != gt.shape[:2]:\n            hazy = cv2.resize(hazy, (gt.shape[1], gt.shape[0]), interpolation=cv2.INTER_AREA)\n        h, w = gt.shape[:2]\n        side = CFG["max_side"]\n        scale = min(1.0, side / max(h, w)) if side > 0 else 1.0\n        if scale < 1:\n            size = (max(1, round(w * scale)), max(1, round(h * scale)))\n            hazy, gt = [cv2.resize(img, size, interpolation=cv2.INTER_AREA) for img in (hazy, gt)]\n        tensor = torch.from_numpy(hazy.copy()).permute(2, 0, 1).float().div_(255)\n        return tensor, gt, name\n\ndef collate(samples):\n    shapes = [item[0].shape[-2:] for item in samples]\n    h = math.ceil(max(s[0] for s in shapes) / 32) * 32\n    w = math.ceil(max(s[1] for s in shapes) / 32) * 32\n    tensors = []\n    for x, _, _ in samples:\n        dh, dw = h - x.shape[-2], w - x.shape[-1]\n        mode = "reflect" if dh < x.shape[-2] and dw < x.shape[-1] else "replicate"\n        tensors.append(F.pad(x, (0, dw, 0, dh), mode=mode))\n    return torch.stack(tensors), [s[1] for s in samples], [s[2] for s in samples], shapes\n\nclass LFRNOnly(nn.Module):\n    def __init__(self, restoration):\n        super().__init__()\n        self.r = restoration\n    def forward(self, ll):\n        n = self.r\n        x0 = n.down_group0(n.layer0(ll))\n        x1 = n.down_group1(n.layer1(x0))\n        x2 = n.down_group2(n.layer2(x1))\n        x3 = n.down_group3(n.layer3(x2))\n        x4 = n.down_group4(n.layer4(x3))\n        y3 = n.up_group4(n.layer_3(x4, x3))\n        y2 = n.up_group3(n.layer_2(y3, x2))\n        y1 = n.up_group2(n.layer_1(y2, x1))\n        y0 = n.up_group1(n.layer_0(y1, x0))\n        return n.last(y0) + ll\n\ndef staged_forward(model, x, dwt, lfrn, iwt):\n    # Events trên cùng stream; không synchronize giữa các giai đoạn.\n    events = [torch.cuda.Event(enable_timing=True) for _ in range(5)]\n    events[0].record()\n    coefficients = dwt(x)\n    events[1].record()\n    count = x.shape[0]\n    ll = lfrn(coefficients[:count])\n    events[2].record()\n    coarse = iwt(torch.cat((ll, coefficients[count:]), dim=0))\n    events[3].record()\n    out = model.restoration_network.DE(coarse)\n    events[4].record()\n    events[-1].synchronize()\n    stages = {name: events[i].elapsed_time(events[i + 1])\n              for i, name in enumerate(("DWT", "LFRN", "IWT", "DEN"))}\n    return out, stages\n\ndef original_forward(model, x):\n    start, end = [torch.cuda.Event(enable_timing=True) for _ in range(2)]\n    wall = time.perf_counter()\n    start.record()\n    out = model(x)\n    end.record()\n    end.synchronize()\n    return out, start.elapsed_time(end), 1000 * (time.perf_counter() - wall)\n\ndef complexity(module, example, stage):\n    # Chỉ cộng phần thop bỏ sót; Conv/Linear đã được thop đếm riêng.\n    extras = defaultdict(float)\n    handles = []\n    original_training = {m: m.training for m in module.modules()}\n    original_hooks = {m: set(m._forward_hooks) for m in module.modules()}\n    original_buffers = {m: set(m._buffers) for m in module.modules()}\n\n    def extra_hook(m, args, output):\n        x = args[0]\n        if isinstance(m, SS2D):\n            b, h, w, _ = x.shape\n            length, d, n, rank, k = h * w, m.d_inner, m.d_state, m.dt_rank, 4\n            extras["einsum"] += 2 * b * k * length * d * (rank + 2 * n + rank)\n            # Recurrence xấp xỉ, coi exp là một phép; không phản ánh chi phí kernel.\n            extras["selective_scan"] += b * k * length * d * (8 * n + 2)\n        elif isinstance(m, FFTBench):\n            b, c, h, w = x.shape\n            # Hai FFT theo upper-style complex FFT estimate; rFFT có thể ít phép hơn.\n            extras["fft_pair"] += 10 * b * c * h * w * math.log2(h * w)\n        elif isinstance(m, DWTForward):\n            extras["internal_haar"] += 4 * x.numel()\n\n    for m in module.modules():\n        if isinstance(m, (SS2D, FFTBench, DWTForward)):\n            handles.append(m.register_forward_hook(extra_hook))\n    try:\n        with torch.inference_mode():\n            macs, _ = profile(module, inputs=(example,), verbose=False)\n    finally:\n        for handle in handles:\n            handle.remove()\n        # thop có thể để lại hook/buffer nếu profiling thất bại.\n        for m in module.modules():\n            m.training = original_training[m]\n            for key in set(m._forward_hooks) - original_hooks[m]:\n                del m._forward_hooks[key]\n            for key in set(m._buffers) - original_buffers[m]:\n                del m._buffers[key]\n    counted = 2 * float(macs)\n    supplemental = sum(extras.values())\n    return {"stage": stage, "counted_GFLOPs": counted / 1e9,\n            "supplemental_estimate_GFLOPs": supplemental / 1e9,\n            "estimated_GFLOPs_per_image": (counted + supplemental) / 1e9,\n            "supplemental_details": dict(extras)}\n\ndef shape_complexity(model, h, w):\n    x = torch.zeros(1, 3, h, w, device="cuda")\n    ll = torch.zeros(1, 3, h // 2, w // 2, device="cuda")\n    rows = []\n    transform = 4 * x.numel() / 1e9  # Scaling + ba phép cộng/trừ trên mỗi hệ số.\n    for stage in ("DWT", "IWT"):\n        rows.append({"stage": stage, "counted_GFLOPs": 0.0,\n                     "supplemental_estimate_GFLOPs": transform,\n                     "estimated_GFLOPs_per_image": transform,\n                     "supplemental_details": {"haar": transform * 1e9}})\n    rows.append(complexity(LFRNOnly(model.restoration_network).eval(), ll, "LFRN"))\n    rows.append(complexity(model.restoration_network.DE, x, "DEN"))\n    rows.sort(key=lambda row: ("DWT", "LFRN", "IWT", "DEN").index(row["stage"]))\n    for row in rows:\n        row.update({"height": h, "width": w,\n                    "estimated_GFLOPs_configured_batch": 16 * row["estimated_GFLOPs_per_image"]})\n    return rows\n\ndef benchmark_dataset(model, label, root, flop_cache):\n    dwt, lfrn, iwt = DWT().eval(), LFRNOnly(model.restoration_network).eval(), IWT().eval()\n    pairs = collect_pairs(root)\n    loader = DataLoader(PairedDataset(pairs), batch_size=CFG["batch_size"], shuffle=False,\n        num_workers=CFG["num_workers"], pin_memory=True, collate_fn=collate,\n        persistent_workers=CFG["num_workers"] > 0, drop_last=False)\n    first = next(iter(loader))[0].cuda()\n    warm_shapes = set()\n    batch_rows, image_rows = [], []\n\n    with torch.inference_mode():\n        for _ in range(CFG["warmup"]):\n            model(first)\n        torch.cuda.synchronize()\n        reference = model(first)\n        staged, _ = staged_forward(model, first, dwt, lfrn, iwt)\n        torch.testing.assert_close(staged, reference, rtol=1e-5, atol=1e-5)\n    del first, reference, staged\n    print(label, ": output phân rã khớp forward gốc.", flush=True)\n\n    iterator = iter(loader)\n    for index in tqdm(range(len(loader)), desc=label):\n        pipeline_start = time.perf_counter()\n        cpu, gt, names, shapes = next(iterator)\n        wait_ms = 1000 * (time.perf_counter() - pipeline_start)\n        t = time.perf_counter()\n        x = cpu.cuda(non_blocking=True)\n        torch.cuda.synchronize()\n        h2d_ms = 1000 * (time.perf_counter() - t)\n        b, _, h, w = x.shape\n        key = (b, h, w)\n        if key not in warm_shapes:\n            with torch.inference_mode():\n                for _ in range(CFG["warmup"]):\n                    model(x)\n            torch.cuda.synchronize()\n            warm_shapes.add(key)\n        torch.cuda.reset_peak_memory_stats()\n        with torch.inference_mode():\n            times, walls, stages = [], [], []\n            for _ in range(CFG["timing_repeats"]):\n                out, gpu_ms, wall_ms = original_forward(model, x)\n                times.append(gpu_ms)\n                walls.append(wall_ms)\n                if len(times) < CFG["timing_repeats"]:\n                    del out\n            t = time.perf_counter()\n            outputs = (out.clamp(0, 1).permute(0, 2, 3, 1).mul(255).round()\n                       .to(torch.uint8).cpu().numpy())\n            d2h_ms = 1000 * (time.perf_counter() - t)\n            del out\n            for _ in range(CFG["timing_repeats"]):\n                stage_out, stage_times = staged_forward(model, x, dwt, lfrn, iwt)\n                stages.append(stage_times)\n                del stage_out\n        peak = torch.cuda.max_memory_allocated() / 1024 ** 3\n        peak_reserved = torch.cuda.max_memory_reserved() / 1024 ** 3\n        row = {"dataset": label, "batch_index": index, "actual_batch_size": b,\n               "padded_height": h, "padded_width": w,\n               "original_forward_gpu_ms": float(np.mean(times)),\n               "original_forward_wall_ms": float(np.mean(walls)),\n               "dataloader_wait_ms": wait_ms, "h2d_ms": h2d_ms, "d2h_ms": d2h_ms,\n               "peak_allocated_GiB": peak, "peak_reserved_GiB": peak_reserved}\n        for name in ("DWT", "LFRN", "IWT", "DEN"):\n            row[name + "_ms"] = float(np.mean([s[name] for s in stages]))\n        row["stage_sum_ms"] = sum(row[n + "_ms"] for n in ("DWT", "LFRN", "IWT", "DEN"))\n        for name, shape, pred, target in zip(names, shapes, outputs, gt):\n            pred = pred[:shape[0], :shape[1]]\n            if min(pred.shape[:2]) < 11:\n                raise ValueError("Ảnh quá nhỏ để tính SSIM window 11.")\n            image_rows.append({"dataset": label, "image_id": name,\n                "height": shape[0], "width": shape[1],\n                "PSNR": float(peak_signal_noise_ratio(target, pred, data_range=255)),\n                "SSIM": float(structural_similarity(target, pred, channel_axis=-1,\n                    data_range=255, gaussian_weights=True, sigma=1.5,\n                    use_sample_covariance=False))})\n        if (h, w) not in flop_cache:\n            flop_cache[(h, w)] = shape_complexity(model, h, w)\n        row["estimated_GFLOPs_per_image"] = sum(\n            r["estimated_GFLOPs_per_image"] for r in flop_cache[(h, w)])\n        row["estimated_GFLOPs_actual_batch"] = b * row["estimated_GFLOPs_per_image"]\n        batch_rows.append(row)\n        # Lưu tiến độ sau từng batch để không mất kết quả khi Colab ngắt.\n        pd.DataFrame(batch_rows).to_csv(OUTPUT / f"{label}_batches.csv", index=False)\n        pd.DataFrame(image_rows).to_csv(OUTPUT / f"{label}_images.csv", index=False)\n        del x, cpu, outputs\n\n    batches, images = pd.DataFrame(batch_rows), pd.DataFrame(image_rows)\n    total = len(images)\n    gpu_ms = batches.original_forward_gpu_ms.sum()\n    wall_ms = batches.original_forward_wall_ms.sum()\n    summary = {"dataset": label, "images": total, "configured_batch_size": CFG["batch_size"],\n        "last_batch_size": int(batches.actual_batch_size.iloc[-1]),\n        "max_side": CFG["max_side"], "PSNR": float(images.PSNR.mean()),\n        "SSIM": float(images.SSIM.mean()), "model_gpu_FPS": total * 1000 / gpu_ms,\n        "model_wall_FPS": total * 1000 / wall_ms,\n        "original_forward_ms_per_image": gpu_ms / total,\n        "original_forward_median_ms_per_batch": float(batches.original_forward_gpu_ms.median()),\n        "original_forward_p95_ms_per_batch": float(batches.original_forward_gpu_ms.quantile(.95)),\n        "peak_allocated_GiB": float(batches.peak_allocated_GiB.max()),\n        "peak_reserved_GiB": float(batches.peak_reserved_GiB.max()),\n        "estimated_GFLOPs_mean_per_image": float(\n            (batches.estimated_GFLOPs_per_image * batches.actual_batch_size).sum() / total)}\n    for name in ("DWT", "LFRN", "IWT", "DEN"):\n        summary[name + "_ms_per_image"] = float(batches[name + "_ms"].sum() / total)\n    return summary\n\ndef main():\n    model, blocks = load_model()\n    print(\'GPU:\', torch.cuda.get_device_name(0), \'| Precision: FP32 | DEN:\', blocks, flush=True)\n    metadata = {"config": CFG, "checkpoint": INPUTS["checkpoint"],\n        "protocol": "RESIDE-6K test split; native DWT/LFRN/IWT/DEN profile",\n        "torch": torch.__version__, "cuda": torch.version.cuda,\n        "gpu": torch.cuda.get_device_name(0), "precision": "FP32; TF32 disabled",\n        "repo_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip(),\n        "params_M": sum(p.numel() for p in model.parameters()) / 1e6,\n        "LFRN_params_M": sum(p.numel() for n, p in model.restoration_network.named_parameters()\n                                if not n.startswith("DE.")) / 1e6,\n        "DEN_params_M": sum(p.numel() for p in model.restoration_network.DE.parameters()) / 1e6,\n        "flops_note": "thop 2*MACs + analytic scan/FFT/wavelet estimates; incomplete elementwise coverage",\n        "metric_note": "RGB uint8; Gaussian SSIM sigma=1.5, population covariance, no crop border"}\n    flop_cache = {}\n    size = CFG["profile_size"]\n    assert size >= 32 and size % 32 == 0\n    flop_cache[(size, size)] = shape_complexity(model, size, size)\n    summaries = []\n    try:\n        for label in ("RESIDE-6K-test",):\n            summaries.append(benchmark_dataset(model, label, Path(INPUTS["datasets"][label]), flop_cache))\n            pd.DataFrame(summaries).to_csv(OUTPUT / "summary.csv", index=False)\n    finally:\n        rows = [row for items in flop_cache.values() for row in items]\n        pd.DataFrame(rows).to_csv(OUTPUT / "stage_gflops.csv", index=False)\n        (OUTPUT / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")\n    print(pd.DataFrame(summaries).to_string(index=False), flush=True)\n    print(\'Kết quả:\', OUTPUT, flush=True)\n\nif __name__ == "__main__":\n    try:\n        main()\n    except torch.cuda.OutOfMemoryError:\n        print(\'Thiếu VRAM ở batch 16. Giữ batch 16, giảm MAX_SIDE hoặc chọn GPU có VRAM lớn hơn.\', flush=True)\n        raise\n').strip() + "\n"
ast.parse(RUNNER_SOURCE)
print("Runner syntax: PASS")

## 6. Run the Phase 1 benchmark

Run once with `LIMIT = 16` to validate the environment. Then change `LIMIT = 0` in the configuration cell and rerun this cell to process the complete RESIDE-6K test split.

In [ ]:
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
run_python("phase1_baseline_runner", RUNNER_SOURCE)

## 7. Required report and Gate G1

This cell writes `P1_baseline_report.md`, `summary.csv`, `batches.csv`, `image_metrics.csv`, `stage_gflops.csv`, and `metadata.json`. Gate G1 is `PASS` only when quality is within tolerance of the published result and DEN is confirmed as the dominant native stage; otherwise the next phase must review or revise the protocol.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display

summary = pd.read_csv(OUTPUT_ROOT / "summary.csv")
batches = pd.read_csv(OUTPUT_ROOT / "RESIDE-6K-test_batches.csv")
images = pd.read_csv(OUTPUT_ROOT / "RESIDE-6K-test_images.csv")
batches.to_csv(OUTPUT_ROOT / "batches.csv", index=False)
images.to_csv(OUTPUT_ROOT / "image_metrics.csv", index=False)
stage = pd.read_csv(OUTPUT_ROOT / "stage_gflops.csv")
row = summary.iloc[0]
den_share = float(row["DEN_ms_per_image"]) / max(1e-12, sum(float(row[f"{n}_ms_per_image"]) for n in ("DWT", "LFRN", "IWT", "DEN")))
psnr_ok = abs(float(row["PSNR"]) - CONFIG["paper_psnr"]) <= CONFIG["psnr_tolerance"]
ssim_ok = abs(float(row["SSIM"]) - CONFIG["paper_ssim"]) <= CONFIG["ssim_tolerance"]
den_ok = den_share >= CONFIG["den_share_threshold"]
gate = "PASS" if psnr_ok and ssim_ok and den_ok else "REVIEW"
metadata_path = OUTPUT_ROOT / "metadata.json"
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
metadata["gate_g1"] = {"status": gate, "psnr_ok": psnr_ok, "ssim_ok": ssim_ok, "den_bottleneck_ok": den_ok, "den_share_native": den_share}
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

lines = ["# P1 Baseline Report", "", "## Gate G1", "", f"Status: {gate}", "",
         "| Check | Observed | Criterion | Result |", "|---|---:|---:|---|",
         f"| PSNR | {float(row['PSNR']):.4f} | {CONFIG['paper_psnr']:.4f} +/- {CONFIG['psnr_tolerance']:.4f} | {'PASS' if psnr_ok else 'REVIEW'} |",
         f"| SSIM | {float(row['SSIM']):.6f} | {CONFIG['paper_ssim']:.6f} +/- {CONFIG['ssim_tolerance']:.6f} | {'PASS' if ssim_ok else 'REVIEW'} |",
         f"| DEN staged latency share | {den_share:.2%} | >= {CONFIG['den_share_threshold']:.0%} | {'PASS' if den_ok else 'REVIEW'} |",
         "", "## Quality and efficiency", "", "| Metric | Value |", "|---|---:|",
         f"| GPU latency / image | {float(row['original_forward_ms_per_image']):.4f} ms |",
         f"| GPU FPS | {float(row['model_gpu_FPS']):.4f} |",
         f"| Peak allocated VRAM | {float(row['peak_allocated_GiB']):.4f} GiB |",
         f"| Estimated FLOPs / image | {float(row['estimated_GFLOPs_mean_per_image']):.4f} G |",
         "", "## Native stage profile", "", "| Stage | ms/image | Share |", "|---|---:|---:|"]
total = sum(float(row[f"{n}_ms_per_image"]) for n in ("DWT", "LFRN", "IWT", "DEN"))
for name in ("DWT", "LFRN", "IWT", "DEN"):
    value = float(row[f"{name}_ms_per_image"])
    lines.append(f"| {name} | {value:.4f} | {value / total:.2%} |")
lines += ["", "## Reproducibility", "", f"- Repository commit: {metadata.get('repo_commit')}", f"- Checkpoint: {metadata.get('checkpoint')}", f"- GPU: {metadata.get('gpu')}", f"- PyTorch/CUDA: {metadata.get('torch')} / {metadata.get('cuda')}", f"- Batch size: {CONFIG['batch_size']}", f"- Max side: {CONFIG['max_side']}", "", "## Decision", "", "Proceed to Phase 2 only after every mismatch is explained and all CSV/JSON artifacts are retained."]
(OUTPUT_ROOT / "P1_baseline_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
display(summary)
display(Markdown((OUTPUT_ROOT / "P1_baseline_report.md").read_text(encoding="utf-8")))

## 8. Package the results

The ZIP contains every report and raw measurement. Download it and place the ZIP under `solution/WDMamba/phase 1/` so the next review can inspect the actual run.

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(str(WORK_ROOT / "wdmamba_phase1_results"), "zip", OUTPUT_ROOT)
print("Created:", archive)
files.download(archive)

## Notes

- The notebook evaluates the published checkpoint; it does not retrain WDMamba.
- `MAX_SIDE = 0` preserves the source resolution. If VRAM is insufficient, change it explicitly and record the change in the report.
- FLOPs are estimates for wavelet/FFT/selective-scan operations; latency is measured independently with CUDA events.
- The final ZIP should include `P1_baseline_report.md`, `metadata.json`, `summary.csv`, `batches.csv`, `image_metrics.csv`, and `stage_gflops.csv`.